# Dermatology Disease Classification

Six erythemato-squamous skin diseases look similar on examination, but 34 clinical and histopathological features — erythema, scaling, itching, koebner phenomenon, melanin incontinence, fibrosis and more — separate them. Predict the disease from those features.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Dermatology (id 33)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/33/data.csv')))
print(df.shape)
print('Patients per disease code:')
print(df['class'].value_counts().sort_index())
print('Missing values per column (only age is affected):')
print(df.isna().sum()[df.isna().sum() > 0])
df.head(3)


(366, 35)
Patients per disease code:
class
1    112
2     61
3     72
4     49
5     52
6     20
Name: count, dtype: int64
Missing values per column (only age is affected):
age    8
dtype: int64


,erythema,scaling,definite-borders,itching,koebner phenomenon,polygonal papules,follicular papules,oral-mucosal involvement,knee elbow involvement,scalp involvement,...,disappearance of the granular layer,vacuolisation and damage of the basal layer,spongiosis,saw-tooth appearance of retes,follicular horn plug,perifollicular parakeratosis,inflammatory monoluclear infiltrate,band-like infiltrate,age,class
0,2,2,0,3,0,0,0,0,1,0,...,0,0,3,0,0,0,1,0,55.0,2
1,3,3,3,2,1,0,0,0,1,1,...,0,0,0,0,0,0,1,0,8.0,1
2,2,1,2,3,1,3,0,3,0,0,...,0,2,3,2,0,0,2,3,26.0,3


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
# 8 missing ages: fill with the TRAINING median, so no test information leaks.
med = Xtr['age'].median()
Xtr = Xtr.fillna({'age': med}); Xte = Xte.fillna({'age': med})
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9674
Logistic Regression accuracy: 0.9674
              precision    recall  f1-score   support

           1       1.00      1.00      1.00        28
           2       0.93      0.88      0.90        16
           3       1.00      1.00      1.00        18
           4       0.85      0.92      0.88        12
           5       1.00      1.00      1.00        13
           6       1.00      1.00      1.00         5

    accuracy                           0.97        92
   macro avg       0.96      0.97      0.96        92
weighted avg       0.97      0.97      0.97        92



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which features separate the skin diseases?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


clubbing of the rete ridges                    0.093750
fibrosis of the papillary dermis               0.082474
thinning of the suprapapillary epidermis       0.079199
spongiosis                                     0.061562
koebner phenomenon                             0.060440
elongation of the rete ridges                  0.059484
focal hypergranulosis                          0.045987
saw-tooth appearance of retes                  0.038758
vacuolisation and damage of the basal layer    0.036867
pnl infiltrate                                 0.033784
band-like infiltrate                           0.033545
knee elbow involvement                         0.029680
melanin incontinence                           0.028621
exocytosis                                     0.028190
oral-mucosal involvement                       0.027902
age                                            0.023575
polygonal papules                              0.023193
scalp involvement                              0

Winner: Random Forest
